# NB0: Environment-Only Descriptors (POSCAR-based, 99 compounds)

Structural / environmental descriptors from POSCAR_std only. Build a moderate pool
(periodic-neighbour GRDF with Gaussian/cosine/Bessel bases + AFS), select the best 2-5 by
XGBoost (5-fold search, LOO confirmation), then merge winners with the proven C6 set.

Parameters in cell 6 are set from the geometry analysis in cell 5 (Rc=6 A, vacuum>=13.96 A
on c for all 99, first bond shell 2.0-3.1 A, gamma in {90,118,120}).

Location: `Keshav-DDP/environment/nb0_environment_descriptors.ipynb`

## Cell 1: Configuration (paths mirror nb5, no file-guessing)

In [ ]:
import os

BASE_DIR = os.path.abspath("..")
RASHBA_CSV = os.path.join(BASE_DIR, "Data", "rashba.csv")
VASP_DIR   = os.path.join(BASE_DIR, "Inverse-design", "rashba")   # {Formula}-{uid}/POSCAR_std
RESULTS_DIR = os.path.join(".", "nb0_environment-results")
os.makedirs(RESULTS_DIR, exist_ok=True)

print("PATH CHECK")
for name, path in [("BASE_DIR", BASE_DIR), ("RASHBA_CSV", RASHBA_CSV), ("VASP_DIR", VASP_DIR)]:
    flag = "OK" if os.path.exists(path) else "MISSING"
    print(f"  [{flag:7s}] {name:11s} = {path}")
print(f"  [OUTPUT ] {'RESULTS_DIR':11s} = {RESULTS_DIR}")

## Cell 2: Imports

In [ ]:
import glob
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from pymatgen.core.structure import Structure
from scipy.special import spherical_jn

print("imports OK")

## Cell 3: Load rashba.csv and collapse to 99 compounds

In [ ]:
df = pd.read_csv(RASHBA_CSV)
print(f"rashba.csv : {df.shape[0]} rows, {df.shape[1]} cols")
print(f"columns    : {list(df.columns)}")
print(f"unique uids: {df['uid'].nunique()}   unique formulas: {df['Formula'].nunique()}")

TARGET = "Rashba_parameter"
assert TARGET in df.columns, f"target '{TARGET}' not in columns: {list(df.columns)}"

idx_max = df.groupby("uid")[TARGET].idxmax()
df99 = df.loc[idx_max].reset_index(drop=True)
print(f"\ncollapsed to {len(df99)} compounds (max {TARGET} per uid)")
print(df99[["Formula", "uid", "kpath", TARGET]].head(8).to_string(index=False))

## Cell 4: Load structures from POSCAR_std (functions verbatim from nb5)

In [ ]:
def find_compound_folder(uid, vasp_dir):
    for folder in glob.glob(os.path.join(vasp_dir, "*")):
        folder_name = os.path.basename(folder)
        idx = folder_name.rfind("-")
        if idx != -1 and folder_name[idx + 1:] == uid:
            return folder
    return None


def load_structure(compound_folder):
    std_path = os.path.join(compound_folder, "POSCAR_std")
    if os.path.exists(std_path):
        return Structure.from_file(std_path)
    matches = glob.glob(os.path.join(compound_folder, "ss_2d*POSCAR"))
    if matches:
        return Structure.from_file(matches[0])
    direct = os.path.join(compound_folder, "POSCAR")
    if os.path.exists(direct):
        return Structure.from_file(direct)
    return None


structures = {}
missing, bad_elem = [], []
for uid in df99["uid"]:
    folder = find_compound_folder(uid, VASP_DIR)
    if folder is None:
        missing.append(uid); continue
    s = load_structure(folder)
    if s is None:
        missing.append(uid); continue
    if max(site.specie.Z for site in s) <= 3:
        bad_elem.append(uid)
    structures[uid] = s

print(f"loaded {len(structures)} / {len(df99)} structures")
print(f"missing           : {len(missing)}  {missing[:5]}")
print(f"max_Z<=3 (suspect): {len(bad_elem)}  {bad_elem[:5]}")

## Cell 5: Geometry analysis (diagnostic; sets cell-6 parameters)

In [ ]:
CUTOFFS = (4, 5, 6, 7, 8)

def analyze_geometry(s, cutoffs=CUTOFFS):
    cart = np.array([site.coords for site in s])
    L = np.array(s.lattice.abc)
    _, _, gamma = s.lattice.angles
    ext = cart.max(axis=0) - cart.min(axis=0)
    vac_axis = int(np.argmax(L))
    slab = ext[vac_axis]; vacuum = L[vac_axis] - slab
    in_plane = [L[i] for i in range(3) if i != vac_axis]
    n = len(s)
    nbrs = s.get_all_neighbors(max(cutoffs))
    dists = np.array([nb.nn_distance for site_nbrs in nbrs for nb in site_nbrs])
    counts = {rc: float((dists <= rc).sum()) / n for rc in cutoffs}
    min_bond = float(dists[dists > 0.4].min()) if dists.size else np.nan
    out = dict(n_atoms=n, a=round(L[0],3), b=round(L[1],3), c=round(L[2],3),
               gamma=round(gamma,1), vac_axis=vac_axis, slab_thick=round(slab,3),
               vacuum=round(vacuum,3), inplane_min=round(min(in_plane),3),
               inplane_max=round(max(in_plane),3), min_bond=round(min_bond,3))
    for rc in cutoffs: out[f"avg_nbr_{rc}A"] = round(counts[rc],2)
    return out

rows = []
for uid, s in structures.items():
    d = analyze_geometry(s)
    d["Formula"] = df99.loc[df99["uid"]==uid,"Formula"].iloc[0]; d["uid"] = uid
    rows.append(d)
geo = pd.DataFrame(rows)
print(geo.head(12).to_string(index=False))
print("\n", geo.describe().T[["min","25%","50%","75%","max"]].round(2).to_string())
geo.to_csv(os.path.join(RESULTS_DIR, "geometry_analysis.csv"), index=False)

## Cell 6: Parameters and radial basis functions

All numbers below are read off cell 5:
- `RC = 6.0`: gives 2-3 neighbour shells per atom, and the smallest vacuum is 13.96 A on c,
  so 6 A never reaches a periodic slab image.
- Gaussian centers span the first bond shell (~2.0-3.1 A) out to the 3rd shell.
- AFS uses a 4 A angular window so angles are real local bond angles, not long-range.
- Angular harmonics cos t / cos 2t / cos 3t cover the 90 deg (orthorhombic) and 120 deg
  (hexagonal) environments present in the set.

In [ ]:
RC = 6.0                                   # neighbour cutoff (A)
GAUSS_CENTERS = [2.0, 2.6, 3.2, 3.8, 4.4, 5.0]
GAUSS_SIGMA   = 0.5
COS_K    = [np.pi/RC * m for m in (1, 2, 3)]
BESSEL_K = np.pi / RC
BESSEL_N = [0, 1, 2]
AFS_MU, AFS_SIGMA = 2.8, 0.8               # radial envelope on the bond shell
AFS_HARMONICS = [1, 2, 3]                  # cos(m*theta)
AFS_RANG = 4.0                             # angular window (A)
FIRST_SHELL = 3.5                          # for coordination + bond-angle stats (A)

def fc(r, rc=RC):
    """Smooth cosine cutoff."""
    return np.where(r <= rc, 0.5*(np.cos(np.pi*r/rc) + 1.0), 0.0)

def b_gauss(r, mu, sig=GAUSS_SIGMA):
    return np.exp(-((r - mu)**2) / (2*sig**2))

def b_cos(r, k):
    return np.cos(k*r)

def b_bessel(r, n, k=BESSEL_K):
    return spherical_jn(n, k*r)

print("parameters set. RC =", RC, " gauss centers =", GAUSS_CENTERS)

## Cell 7: Periodic neighbour data (the key fix vs the old intra-cell version)

In [ ]:
def neighbor_data(s, rc=RC):
    """Per atom: distances, vectors (atom->neighbour image), neighbour Z, central Z."""
    alln = s.get_all_neighbors(rc)
    out = []
    for i, site in enumerate(s):
        ci = site.coords; zi = site.specie.Z
        d, v, zj = [], [], []
        for nb in alln[i]:
            dist = nb.nn_distance
            if dist <= 1e-3:
                continue
            d.append(dist); v.append(np.array(nb.coords) - ci); zj.append(nb.specie.Z)
        out.append(dict(zi=zi,
                        d=np.array(d),
                        v=np.array(v) if v else np.zeros((0, 3)),
                        zj=np.array(zj)))
    return out

# sanity on one compound
_uid = next(iter(structures))
_nd = neighbor_data(structures[_uid])
print(f"{_uid}: {len(_nd)} atoms, neighbours/atom = {[len(a['d']) for a in _nd]}")

## Cell 8: GRDF, AFS, and simple geometric features

In [ ]:
def grdf(nd, basis, weighted=False):
    """Per-atom sum of basis(r)*fc(r), aggregated over atoms (mean, std)."""
    per = []
    for a in nd:
        d = a["d"]
        if d.size == 0:
            per.append(0.0); continue
        w = basis(d) * fc(d)
        if weighted:
            w = w * np.sqrt(a["zi"] * a["zj"])
        per.append(float(w.sum()))
    per = np.array(per)
    return per.mean(), per.std()

def afs(nd):
    """Per-atom angular sum over neighbour pairs within AFS_RANG, mean+std over atoms."""
    res = {m: [] for m in AFS_HARMONICS}
    for a in nd:
        d, v = a["d"], a["v"]
        mask = d <= AFS_RANG
        d, v = d[mask], v[mask]
        if d.size < 2:
            for m in AFS_HARMONICS: res[m].append(0.0)
            continue
        wr = np.exp(-((d - AFS_MU)**2) / (2*AFS_SIGMA**2)) * fc(d)
        acc = {m: 0.0 for m in AFS_HARMONICS}
        for j in range(len(d)):
            for k in range(j+1, len(d)):
                c = np.clip(np.dot(v[j], v[k]) / (d[j]*d[k]), -1, 1)
                th = np.arccos(c); ww = wr[j]*wr[k]
                for m in AFS_HARMONICS:
                    acc[m] += ww * np.cos(m*th)
        for m in AFS_HARMONICS: res[m].append(acc[m])
    return {m: (np.array(res[m]).mean(), np.array(res[m]).std()) for m in AFS_HARMONICS}

def simple_geom(nd):
    coord, bonds, angles = [], [], []
    for a in nd:
        d, v = a["d"], a["v"]
        m = d <= FIRST_SHELL
        coord.append(int(m.sum()))
        dd, vv = d[m], v[m]
        bonds.extend(dd.tolist())
        for j in range(len(dd)):
            for k in range(j+1, len(dd)):
                c = np.clip(np.dot(vv[j], vv[k]) / (dd[j]*dd[k]), -1, 1)
                angles.append(np.degrees(np.arccos(c)))
    coord = np.array(coord); bonds = np.array(bonds); angles = np.array(angles)
    return dict(coord_num=coord.mean(),
                bond_mean=bonds.mean() if bonds.size else 0.0,
                bond_std=bonds.std() if bonds.size else 0.0,
                angle_mean=angles.mean() if angles.size else 0.0,
                angle_std=angles.std() if angles.size else 0.0)

print("descriptor functions defined")

## Cell 9: Build the descriptor table (core + flagged Z-weighted)

In [ ]:
def features_for(s):
    nd = neighbor_data(s)
    feat = {}
    # GRDF Gaussian (core) + Z-weighted (zw_)
    for mu in GAUSS_CENTERS:
        mn, sd = grdf(nd, lambda r, mu=mu: b_gauss(r, mu))
        feat[f"grdf_gauss_{mu:.1f}_mean"] = mn; feat[f"grdf_gauss_{mu:.1f}_std"] = sd
        zmn, zsd = grdf(nd, lambda r, mu=mu: b_gauss(r, mu), weighted=True)
        feat[f"zw_grdf_gauss_{mu:.1f}_mean"] = zmn; feat[f"zw_grdf_gauss_{mu:.1f}_std"] = zsd
    # GRDF cosine
    for i, k in enumerate(COS_K, 1):
        mn, sd = grdf(nd, lambda r, k=k: b_cos(r, k))
        feat[f"grdf_cos_k{i}_mean"] = mn; feat[f"grdf_cos_k{i}_std"] = sd
    # GRDF Bessel
    for n in BESSEL_N:
        mn, sd = grdf(nd, lambda r, n=n: b_bessel(r, n))
        feat[f"grdf_bessel_n{n}_mean"] = mn; feat[f"grdf_bessel_n{n}_std"] = sd
    # AFS
    a = afs(nd)
    for m in AFS_HARMONICS:
        feat[f"afs_m{m}_mean"], feat[f"afs_m{m}_std"] = a[m]
    # simple geometric
    feat.update(simple_geom(nd))
    return feat

recs = []
for i, uid in enumerate(df99["uid"]):
    if uid not in structures:
        continue
    f = features_for(structures[uid]); f["uid"] = uid
    recs.append(f)
    if (i+1) % 25 == 0:
        print(f"  {i+1}/{len(df99)}")

df_env = pd.DataFrame(recs)
df_env.to_csv(os.path.join(RESULTS_DIR, "environment_descriptors_99.csv"), index=False)
print(f"\ndf_env: {df_env.shape[0]} rows, {df_env.shape[1]-1} features")
core_cols = [c for c in df_env.columns if c not in ("uid",) and not c.startswith("zw_")]
zw_cols   = [c for c in df_env.columns if c.startswith("zw_")]
print(f"core features: {len(core_cols)}   zw features: {len(zw_cols)}")

## Cell 10: Modeling setup (XGBoost, same params as prior stages)

Search uses 5-fold CV (fast). Final reported number for each chosen set is true LOO,
matching your `eval_reg_99`.

In [ ]:
from sklearn.model_selection import LeaveOneOut, KFold, cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor

XGB_REG_PARAMS = dict(n_estimators=100, max_depth=3, learning_rate=0.1,
                      subsample=0.8, colsample_bytree=0.8,
                      reg_alpha=1.0, reg_lambda=1.0, random_state=42, verbosity=0)

work = df_env.merge(df99[["uid", TARGET]], on="uid").reset_index(drop=True)
y = work[TARGET].values
POOL_CORE = [c for c in core_cols]
POOL_ALL  = core_cols + zw_cols
print(f"aligned: {len(work)} compounds, target {TARGET}")
print(f"core pool: {len(POOL_CORE)} features")

_kf = KFold(n_splits=5, shuffle=True, random_state=42)
def eval_cv(feats):
    X = work[feats].fillna(0).values
    yp = cross_val_predict(XGBRegressor(**XGB_REG_PARAMS), X, y, cv=_kf)
    return r2_score(y, yp), mean_absolute_error(y, yp)

def eval_loo(feats):
    X = work[feats].fillna(0).values
    yp = cross_val_predict(XGBRegressor(**XGB_REG_PARAMS), X, y, cv=LeaveOneOut())
    return r2_score(y, yp), mean_absolute_error(y, yp)

print("baseline single-feature spot check (5-fold):", round(eval_cv([POOL_CORE[0]])[0], 3))

## Cell 11: Phase 1 - exhaustive single features (core pool)

In [ ]:
import itertools
singles = sorted(((eval_cv([f])[0], f) for f in POOL_CORE), reverse=True)
print("top 12 single features (5-fold R2):")
for r2, f in singles[:12]:
    print(f"  {r2:6.3f}  {f}")
best1 = [singles[0][1]]
pd.DataFrame(singles, columns=["cv_r2", "feature"]).to_csv(
    os.path.join(RESULTS_DIR, "select_singles.csv"), index=False)

## Cell 12: Phase 2 - exhaustive pairs (core pool)

In [ ]:
pairs = []
for a, b in itertools.combinations(POOL_CORE, 2):
    pairs.append((eval_cv([a, b])[0], a, b))
pairs.sort(reverse=True)
print("top 12 pairs (5-fold R2):")
for r2, a, b in pairs[:12]:
    print(f"  {r2:6.3f}  {a} + {b}")
best2 = [pairs[0][1], pairs[0][2]]
pd.DataFrame(pairs, columns=["cv_r2", "f1", "f2"]).to_csv(
    os.path.join(RESULTS_DIR, "select_pairs.csv"), index=False)

## Cell 13: Phase 3 - forward select 3 to 5, then confirm with LOO

In [ ]:
def forward(seed, pool, upto):
    chosen = list(seed); steps = []
    while len(chosen) < upto:
        cand = [(eval_cv(chosen + [f])[0], f) for f in pool if f not in chosen]
        cand.sort(reverse=True)
        chosen.append(cand[0][1]); steps.append((len(chosen), list(chosen), cand[0][0]))
    return steps

steps = forward(best2, POOL_CORE, upto=5)
best_by_n = {1: best1, 2: best2}
for n, feats, _ in steps:
    best_by_n[n] = feats

print("BEST SET PER SIZE (core pool)")
print(f"{'n':>2}  {'cv_R2':>7}  {'LOO_R2':>7}  {'LOO_MAE':>7}  features")
summary = []
for n in sorted(best_by_n):
    feats = best_by_n[n]
    cvr = eval_cv(feats)[0]; lr, lm = eval_loo(feats)
    summary.append(dict(n=n, cv_r2=round(cvr,3), loo_r2=round(lr,3),
                        loo_mae=round(lm,3), features=", ".join(feats)))
    print(f"{n:>2}  {cvr:7.3f}  {lr:7.3f}  {lm:7.3f}  {', '.join(feats)}")
pd.DataFrame(summary).to_csv(os.path.join(RESULTS_DIR, "select_summary_core.csv"), index=False)

## Cell 14: Does Z-weighting add anything on top of the core winner?

In [ ]:
base = best_by_n[max(best_by_n)]
base_r2 = eval_loo(base)[0]
print(f"core best ({len(base)} feat) LOO R2 = {base_r2:.3f}")
probe = sorted(((eval_loo(base + [z])[0], z) for z in zw_cols), reverse=True)
print("best zw additions (LOO R2 of core_best + one zw feature):")
for r2, z in probe[:8]:
    print(f"  {r2:6.3f}  ({r2-base_r2:+.3f})  +{z}")

## Cell 15: Merge environment winners with the proven C6 set

VERIFY `C6_CSV` below. It must be the 99-row CSV that holds the C6 feature columns per uid.
The path is a guess from the directory tree; if your C6 values live elsewhere, set it here.
The cell prints columns and asserts the 6 C6 features resolve before running.

In [ ]:
C6_CSV = os.path.join(BASE_DIR, "k-path", "old+new_nb6",
                      "nb7_combined-results", "merged_old_new_99.csv")   # VERIFY
print("C6_CSV exists:", os.path.exists(C6_CSV))

def resolve(name, cols):
    if name in cols: return name
    if f"old_{name}" in cols: return f"old_{name}"
    if f"new_{name}" in cols: return f"new_{name}"
    raise KeyError(name)

if os.path.exists(C6_CSV):
    dc6 = pd.read_csv(C6_CSV)
    print("columns:", list(dc6.columns)[:40], "..." if dc6.shape[1] > 40 else "")
    C6_WANT = ["E_pfrac_VBM", "E_pfrac_CBM", "radius_mean",
               "pmid_afs_gauss_std", "kpath_angle_deg", "ehull"]
    C6 = [resolve(c, dc6.columns) for c in C6_WANT]
    keep = ["uid"] + C6
    merged = work.merge(dc6[keep].drop_duplicates("uid"), on="uid")
    yy = merged[TARGET].values

    def loo(feats):
        X = merged[feats].fillna(0).values
        yp = cross_val_predict(XGBRegressor(**XGB_REG_PARAMS), X, yy, cv=LeaveOneOut())
        return r2_score(yy, yp), mean_absolute_error(yy, yp)

    print(f"\nC6 alone               LOO R2 = {loo(C6)[0]:.3f}")
    for n in (2, 3, 4, 5):
        if n in best_by_n:
            add = best_by_n[n]
            print(f"C6 + top-{n} env ({n} feat) LOO R2 = {loo(C6 + add)[0]:.3f}   added: {', '.join(add)}")
else:
    print("set C6_CSV to the correct path and re-run this cell")